# Landing Zone
Secuencia de la fase:
1. Configuracion de rutas y datasets raw.
2. Utilidades de trazabilidad y escritura de metadatos.
3. Copia versionada de los ficheros originales.
4. Ejecucion y verificacion del manifiesto.


In [1]:
import json
import shutil
from datetime import UTC, datetime
from pathlib import Path

ROOT = Path.cwd()
LANDING_ZONE = ROOT / "landing_zone"
LANDING_RAW = LANDING_ZONE / "raw"
LANDING_REPORTS = LANDING_ZONE / "reports"
DATASETS = {
    "pics": {"path": ROOT / "opendatabcn_pics-csv.csv", "encoding": "utf-16"},
    "weather": {"path": ROOT / "dataset_prat_temps_2025-26.csv", "encoding": "utf-8-sig"},
    "hotels": {"path": ROOT / "opendatabcn_allotjament_hotels-csv.csv", "encoding": "utf-16"},
    "airbnb_listings": {"path": ROOT / "listings.csv", "encoding": "utf-8-sig"},
    "airbnb_neighbourhoods": {"path": ROOT / "neighbourhoods.csv", "encoding": "utf-8-sig"},
    "airbnb_reviews": {"path": ROOT / "reviews_airbnb.csv", "encoding": "utf-8-sig"},
    "airbnb_calendar": {"path": ROOT / "calendar.csv", "encoding": "utf-8-sig"},
    "income_2022": {"path": ROOT / "2022_renda_disponible_llars_per_persona.csv", "encoding": "utf-8-sig"},
    "hut_licenses": {"path": ROOT / "hut_comunicacio_opendata.csv", "encoding": "utf-8-sig"},
}


def ensure_phase_dirs() -> None:
    for path in [LANDING_RAW, LANDING_REPORTS]:
        path.mkdir(parents=True, exist_ok=True)


In [2]:
def utc_run_id() -> str:
    return datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")

def write_json(path: Path, payload: object) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")


In [3]:
def run_collectors(run_id: str) -> dict[str, Path]:
    landing_paths: dict[str, Path] = {}
    for dataset, meta in DATASETS.items():
        source = meta["path"]
        target_dir = LANDING_RAW / dataset
        target_dir.mkdir(parents=True, exist_ok=True)
        target = target_dir / f"{run_id}_{source.name}"
        shutil.copy2(source, target)
        landing_paths[dataset] = target
    write_json(
        LANDING_REPORTS / "landing_manifest.json",
        {
            "run_id": run_id,
            "created_at_utc": datetime.now(UTC).isoformat(),
            "datasets": {name: str(path.relative_to(ROOT)) for name, path in landing_paths.items()},
        },
    )
    return landing_paths


In [4]:
ensure_phase_dirs()
landing_result = run_collectors(utc_run_id())
{name: str(path.relative_to(ROOT)) for name, path in landing_result.items()}


{'pics': 'landing_zone/raw/pics/20260404T112053Z_opendatabcn_pics-csv.csv', 'weather': 'landing_zone/raw/weather/20260404T112053Z_dataset_prat_temps_2025-26.csv', 'hotels': 'landing_zone/raw/hotels/20260404T112053Z_opendatabcn_allotjament_hotels-csv.csv', 'airbnb_listings': 'landing_zone/raw/airbnb_listings/20260404T112053Z_listings.csv', 'airbnb_neighbourhoods': 'landing_zone/raw/airbnb_neighbourhoods/20260404T112053Z_neighbourhoods.csv', 'airbnb_reviews': 'landing_zone/raw/airbnb_reviews/20260404T112053Z_reviews_airbnb.csv', 'airbnb_calendar': 'landing_zone/raw/airbnb_calendar/20260404T112053Z_calendar.csv', 'income_2022': 'landing_zone/raw/income_2022/20260404T112053Z_2022_renda_disponible_llars_per_persona.csv', 'hut_licenses': 'landing_zone/raw/hut_licenses/20260404T112053Z_hut_comunicacio_opendata.csv'}